<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Thresholds and costs

**[C03-M06-L02 · Thresholds and costs](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/)** · C03, Essential Mathematics and Statistics for AI · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** turn the counts at a threshold into the review workload and a total cost, calculate the cost curve at all 101 thresholds, find the cheapest threshold, and see how it moves when the costs or the base rate change.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The confusion matrix, precision and recall, from [Classification trade-offs: Now the names](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/#now-the-names). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M06-L02-thresholds-and-costs/thresholds-and-costs-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the scores of the 1,000 test tickets

The next cell defines two lists: `urgent_scores`, the model's scores of the 80 urgent test tickets, and `normal_scores`, the scores of the 920 tickets that are not urgent. They are the same seeded scores that the lesson's explorer draws. The data is fictional.

Run the cell. You should see `Ready: 80 urgent and 920 not urgent tickets, 1000 in total`.

In [ ]:
# The scores, written in full (you do not need to read them).
urgent_scores = [
    0.828, 0.694, 0.812, 0.634, 0.732, 0.326, 0.654, 0.66, 0.382, 0.674, 0.747, 0.951, 0.867, 0.507, 0.59, 0.678,
    0.469, 0.788, 0.777, 0.379, 0.701, 0.708, 0.505, 0.867, 0.58, 0.578, 0.527, 0.573, 0.684, 0.77, 0.426, 0.566,
    0.55, 0.746, 0.665, 0.538, 0.842, 0.536, 0.593, 0.655, 0.904, 0.551, 0.605, 0.766, 0.727, 0.486, 0.47, 0.693,
    0.526, 0.615, 0.895, 0.844, 0.296, 0.775, 0.308, 0.88, 0.753, 0.576, 0.565, 0.645, 0.372, 0.589, 0.446, 0.555,
    0.517, 0.579, 0.877, 0.707, 0.665, 0.356, 0.993, 0.426, 0.42, 0.633, 0.587, 0.462, 0.824, 0.813, 0.681, 0.575,
]
normal_scores = [
    0.153, 0.358, 0.566, 0.22, 0.364, 0.461, 0.137, 0.223, 0.403, 0.099, 0.213, 0.57, 0.437, 0.369, 0.431, 0.182,
    0.128, 0.525, 0.043, 0.201, 0.281, 0.36, 0.352, 0.282, 0.289, 0.033, 0.344, 0.119, 0.348, 0.586, 0.206, 0.137,
    0.048, 0.303, 0.178, 0.383, 0.478, 0.24, 0.09, 0.302, 0.332, 0.284, 0.291, 0.38, 0.19, 0.103, 0.388, 0.401,
    0.071, 0.371, 0.083, 0.23, 0.104, 0.381, 0.377, 0.048, 0.285, 0.317, 0.425, 0.38, 0.216, 0.13, 0.126, 0.501,
    0.196, 0.424, 0.195, 0.241, 0.653, 0.368, 0.418, 0.062, 0.465, 0.394, 0.455, 0.398, 0.239, 0.056, 0.085, 0.281,
    0.31, 0.255, 0.087, 0.073, 0.337, 0.116, 0.022, 0.415, 0.345, 0.412, 0.357, 0.393, 0.703, 0.524, 0.223, 0.258,
    0.303, 0.03, 0.307, 0.441, 0.153, 0.35, 0.34, 0.303, 0.127, 0.179, 0.257, 0.529, 0.142, 0.294, 0.283, 0.304,
    0.298, 0.236, 0.477, 0.108, 0.052, 0.574, 0.102, 0.456, 0.317, 0.274, 0.219, 0.464, 0.178, 0.381, 0.16, 0.168,
    0.402, 0.241, 0.302, 0.127, 0.326, 0.185, 0.291, 0.392, 0.224, 0.259, 0.443, 0.176, 0.248, 0.712, 0.269, 0.418,
    0.33, 0.084, 0.197, 0.265, 0.21, 0.02, 0.418, 0.42, 0.163, 0.488, 0.211, 0.173, 0.257, 0.258, 0.368, 0.281,
    0.048, 0.427, 0.239, 0.483, 0.158, 0.112, 0.283, 0.355, 0.414, 0.133, 0.244, 0.288, 0.281, 0.113, 0.333, 0.285,
    0.132, 0.274, 0.069, 0.134, 0.087, 0.274, 0.386, 0.468, 0.332, 0.358, 0.436, 0.255, 0.315, 0.311, 0.347, 0.119,
    0.133, 0.186, 0.435, 0.023, 0.092, 0.016, 0.643, 0.364, 0.476, 0.4, 0.056, 0.428, 0.203, 0.391, 0.255, 0.355,
    0.111, 0.102, 0.208, 0.446, 0.054, 0.287, 0.331, 0.404, 0.07, 0.47, 0.233, 0.007, 0.085, 0.32, 0.115, 0.125,
    0.443, 0.412, 0.078, 0.366, 0.186, 0.094, 0.173, 0.268, 0.139, 0.52, 0.167, 0.251, 0.306, 0.105, 0.231, 0.267,
    0.059, 0.236, 0.234, 0.315, 0.142, 0.146, 0.128, 0.228, 0.219, 0.203, 0.388, 0.147, 0.125, 0.441, 0.526, 0.063,
    0.053, 0.401, 0.34, 0.313, 0.32, 0.095, 0.242, 0.18, 0.151, 0.208, 0.312, 0.212, 0.494, 0.341, 0.346, 0.274,
    0.272, 0.321, 0.197, 0.259, 0.009, 0.165, 0.135, 0.207, 0.623, 0.26, 0.134, 0.333, 0.152, 0.316, 0.211, 0.344,
    0.129, 0.334, 0.105, 0.335, 0.176, 0.183, 0.015, 0.206, 0.162, 0.254, 0.386, 0.343, 0.595, 0.218, 0.311, 0.251,
    0.118, 0.139, 0.112, 0.253, 0.282, 0.382, 0.425, 0.426, 0.355, 0.207, 0.403, 0.307, 0.282, 0.252, 0.193, 0.213,
    0.274, 0.189, 0.398, 0.136, 0.153, 0.075, 0.416, 0.61, 0.324, 0.172, 0.212, 0.541, 0.128, 0.214, 0.368, 0.112,
    0.173, 0.108, 0.013, 0.328, 0.16, 0.194, 0.2, 0.231, 0.205, 0.142, 0.26, 0.209, 0.227, 0.118, 0.464, 0.206,
    0.259, 0.649, 0.392, 0.231, 0.202, 0.167, 0.318, 0.255, 0.383, 0.476, 0.519, 0.327, 0.289, 0.153, 0.295, 0.112,
    0.178, 0.302, 0.244, 0.345, 0.487, 0.062, 0.344, 0.296, 0.241, 0.378, 0.213, 0.433, 0.32, 0.462, 0.279, 0.177,
    0.098, 0.434, 0.175, 0.292, 0.077, 0.348, 0.387, 0.348, 0.294, 0.191, 0.169, 0.24, 0.397, 0.092, 0.379, 0.288,
    0.283, 0.308, 0.304, 0.096, 0.269, 0.075, 0.579, 0.195, 0.18, 0.269, 0.213, 0.274, 0.075, 0.069, 0.099, 0.365,
    0.209, 0.19, 0.296, 0.126, 0.066, 0.294, 0.187, 0.338, 0.095, 0.109, 0.481, 0.056, 0.236, 0.081, 0.341, 0.159,
    0.295, 0.261, 0.409, 0.33, 0.304, 0.106, 0.066, 0.267, 0.307, 0.431, 0.19, 0.299, 0.505, 0.38, 0.375, 0.366,
    0.316, 0.12, 0.471, 0.374, 0.421, 0.183, 0.346, 0.258, 0.127, 0.439, 0.308, 0.226, 0.222, 0.237, 0.214, 0.11,
    0.027, 0.197, 0.115, 0.374, 0.298, 0.433, 0.035, 0.061, 0.377, 0.289, 0.26, 0.215, 0.285, 0.006, 0.483, 0.148,
    0.097, 0.121, 0.048, 0.336, 0.152, 0.261, 0.376, 0.315, 0.186, 0.35, 0.217, 0.323, 0.193, 0.043, 0.352, 0.306,
    0.38, 0.25, 0.446, 0.579, 0.097, 0.225, 0.235, 0.462, 0.082, 0.463, 0.287, 0.149, 0.267, 0.291, 0.242, 0.297,
    0.379, 0.165, 0.375, 0.178, 0.37, 0.207, 0.162, 0.32, 0.126, 0.466, 0.139, 0.193, 0.023, 0.437, 0.422, 0.539,
    0.153, 0.297, 0.093, 0.413, 0.255, 0.282, 0.161, 0.377, 0.243, 0.237, 0.009, 0.231, 0.454, 0.274, 0.079, 0.086,
    0.089, 0.116, 0.412, 0.342, 0.324, 0.139, 0.143, 0.23, 0.173, 0.224, 0.186, 0.468, 0.304, 0.104, 0.27, 0.046,
    0.324, 0.317, 0.302, 0.306, 0.338, 0.722, 0.277, 0.248, 0.345, 0.115, 0.315, 0.368, 0.288, 0.128, 0.143, 0.337,
    0.153, 0.312, 0.144, 0.078, 0.363, 0.203, 0.226, 0.207, 0.051, 0.215, 0.184, 0.284, 0.311, 0.285, 0.1, 0.136,
    0.081, 0.221, 0.272, 0.457, 0.117, 0.373, 0.277, 0.248, 0.223, 0.264, 0.067, 0.407, 0.283, 0.34, 0.203, 0.395,
    0.179, 0.134, 0.208, 0.219, 0.352, 0.263, 0.23, 0.178, 0.31, 0.217, 0.369, 0.405, 0.311, 0.308, 0.509, 0.286,
    0.335, 0.061, 0.466, 0.282, 0.242, 0.393, 0.425, 0.08, 0.091, 0.1, 0.11, 0.35, 0.38, 0.168, 0.477, 0.204,
    0.13, 0.234, 0.298, 0.327, 0.219, 0.552, 0.329, 0.404, 0.206, 0.248, 0.353, 0.312, 0.087, 0.269, 0.242, 0.102,
    0.254, 0.337, 0.218, 0.111, 0.212, 0.297, 0.176, 0.17, 0.263, 0.413, 0.542, 0.328, 0.248, 0.304, 0.388, 0.375,
    0.332, 0.029, 0.431, 0.266, 0.025, 0.376, 0.171, 0.373, 0.216, 0.471, 0.343, 0.363, 0.522, 0.285, 0.166, 0.196,
    0.346, 0.182, 0.456, 0.362, 0.336, 0.244, 0.297, 0.151, 0.278, 0.249, 0.411, 0.158, 0.096, 0.065, 0.108, 0.074,
    0.274, 0.512, 0.373, 0.495, 0.513, 0.293, 0.311, 0.102, 0.48, 0.499, 0.339, 0.24, 0.228, 0.47, 0.271, 0.067,
    0.117, 0.3, 0.526, 0.124, 0.413, 0.229, 0.108, 0.248, 0.306, 0.168, 0.209, 0.202, 0.343, 0.36, 0.453, 0.47,
    0.239, 0.19, 0.071, 0.355, 0.125, 0.468, 0.254, 0.219, 0.304, 0.263, 0.262, 0.047, 0.52, 0.18, 0.462, 0.397,
    0.312, 0.244, 0.097, 0.333, 0.24, 0.243, 0.489, 0.28, 0.314, 0.186, 0.051, 0.356, 0.293, 0.143, 0.167, 0.172,
    0.069, 0.137, 0.303, 0.514, 0.329, 0.143, 0.109, 0.201, 0.385, 0.426, 0.204, 0.412, 0.031, 0.363, 0.062, 0.056,
    0.211, 0.348, 0.325, 0.346, 0.168, 0.37, 0.109, 0.257, 0.163, 0.455, 0.364, 0.18, 0.457, 0.162, 0.303, 0.044,
    0.22, 0.275, 0.122, 0.159, 0.352, 0.464, 0.229, 0.121, 0.4, 0.094, 0.358, 0.218, 0.509, 0.37, 0.248, 0.082,
    0.401, 0.26, 0.302, 0.236, 0.127, 0.537, 0.166, 0.009, 0.49, 0.322, 0.217, 0.144, 0.229, 0.12, 0.341, 0.029,
    0.015, 0.107, 0.707, 0.199, 0.508, 0.303, 0.372, 0.225, 0.211, 0.156, 0.217, 0.097, 0.238, 0.125, 0.385, 0.342,
    0.427, 0.146, 0.284, 0.257, 0.289, 0.109, 0.51, 0.254, 0.206, 0.038, 0.146, 0.467, 0.273, 0.519, 0.182, 0.167,
    0.247, 0.554, 0.302, 0.203, 0.411, 0.527, 0.015, 0.244, 0.289, 0.068, 0.04, 0.404, 0.244, 0.131, 0.127, 0.474,
    0.347, 0.171, 0.176, 0.322, 0.23, 0.476, 0.204, 0.221, 0.327, 0.305, 0.076, 0.728, 0.011, 0.535, 0.262, 0.135,
    0.129, 0.136, 0.212, 0.223, 0.071, 0.23, 0.606, 0.009, 0.454, 0.246, 0.198, 0.136, 0.425, 0.101, 0.312, 0.31,
    0.404, 0.24, 0.216, 0.434, 0.325, 0.115, 0.638, 0.578,
]
print(f"Ready: {len(urgent_scores)} urgent and {len(normal_scores)} not urgent tickets, {len(urgent_scores) + len(normal_scores)} in total")

## Setup: count at a threshold

The next cell defines `count(threshold)`. It returns the four counts of the confusion matrix: true positives, false negatives, false positives and true negatives. It also defines `thresholds`, the 101 thresholds from 0.00 to 1.00 in steps of 0.01.

Run the cell. You should see `TP 65, FN 15, FP 47, TN 873`: the counts at threshold 0.50 from [Classification trade-offs](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/#worked-example-the-1000-test-tickets).

In [ ]:
def count(threshold):
    """The four counts at a threshold. A ticket is flagged when its score is at least the threshold."""
    tp = len([s for s in urgent_scores if s >= threshold])   # urgent, flagged
    fp = len([s for s in normal_scores if s >= threshold])   # not urgent, flagged
    fn = len(urgent_scores) - tp                              # urgent, missed
    tn = len(normal_scores) - fp                              # not urgent, correctly left
    return tp, fn, fp, tn

# The 101 thresholds of the explorer: 0.00, 0.01, ..., 1.00
thresholds = [round(i / 100, 2) for i in range(101)]

tp, fn, fp, tn = count(0.50)
print(f"TP {tp}, FN {fn}, FP {fp}, TN {tn}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One threshold, three consequences

The test set has 1,000 tickets, and Larkfield gets about 1,200 tickets a week. If new tickets look like the test tickets, multiply a count per 1,000 by 1.2 to get a count per week. Divide the week by 7 to get a day.

> **Predict.** At threshold 0.30, about how many tickets a day do the senior agents review? Then run the cell.

In [ ]:
rows = []
for t in [0.30, 0.50, 0.70]:
    tp, fn, fp, tn = count(t)
    flagged = tp + fp
    per_week = flagged * 1.2
    rows.append([f"{t:.2f}", flagged, fn, fp, round(per_week), round(per_week / 7)])
table(["Threshold", "Flagged per 1,000", "Missed urgent", "False alarms", "Reviews per week", "Reviews per day"], rows)

> **Check.** You should see 442, 112 and 32 flagged tickets; 530, 134 and 38 reviews a week; and about 76, 19 and 5 reviews a day. At 0.30, most reviews are false alarms (363). At 0.70, 53 of the 80 urgent tickets are missed.

## 2. Give each mistake a cost

Omar gives each mistake a cost in **cost points**. A review of a flagged ticket costs 2 points. A missed urgent ticket costs 30 points. A true negative costs nothing.

total cost = 2 × flagged + 30 × missed

The next cell defines `total_cost(threshold, review, miss)` and calculates five choices.

> **Predict.** Which of the five is the cheapest? Then run the cell.

In [ ]:
def total_cost(threshold, review=2, miss=30):
    """Review cost of every flag plus the cost of every missed urgent ticket, per 1,000 tickets."""
    tp, fn, fp, tn = count(threshold)
    return review * (tp + fp) + miss * fn

rows = []
for name, t in [("Flag everything", 0.00), ("0.30", 0.30), ("0.50", 0.50), ("0.70", 0.70), ("Flag nothing", 1.00)]:
    tp, fn, fp, tn = count(t)
    rows.append([name, tp + fp, fn, 2 * (tp + fp), 30 * fn, total_cost(t)])
table(["Threshold", "Flagged", "Missed", "2 × flagged", "30 × missed", "Total cost"], rows)

> **Check.** You should see the totals 2000, 914, 674, 1654 and 2400. Of these five, 0.50 is the cheapest. Flagging nothing (1.00) costs the most: all 80 urgent tickets are missed.

## 3. The cost curve: all 101 thresholds

The explorer on the lesson page calculates the cost at every threshold from 0.00 to 1.00. Here you do the same with a loop.

> **Predict.** Is the cheapest threshold lower or higher than 0.50? Think about which mistake costs more. Then run the cell.

In [ ]:
costs = [total_cost(t) for t in thresholds]
lowest = min(costs)
cheapest = thresholds[costs.index(lowest)]      # the first threshold with the lowest cost
print("cheapest threshold:", cheapest, "| total cost:", lowest)
print("thresholds that cost less than 700:", [t for t, c in zip(thresholds, costs) if c < 700])
print()
every_5 = thresholds[::5]
bars([f"{t:.2f}" for t in every_5], [total_cost(t) for t in every_5])

> **Check.** You should see the cheapest threshold 0.42, at 606 cost points, and every threshold from 0.37 to 0.50 under 700. The bars fall slowly on the left (each step removes false alarms, at 2 points each) and rise fast on the right (each step misses urgent tickets, at 30 points each). The bars show only every fifth threshold.

The bottom of the curve is wide and a little uneven. Run the cell to see the costs from 0.36 to 0.51. One more missed urgent ticket adds 30 points at once, so the curve has small steps.

In [ ]:
table(["Threshold", "Total cost"], [[f"{t:.2f}", total_cost(t)] for t in thresholds if 0.36 <= t <= 0.51])

> **Check.** You should see costs from 606 (at 0.42) to 696 (at 0.38) between 0.37 and 0.50, and 718 at 0.36 and 720 at 0.51. The test set has only 80 urgent tickets, so report a range: "the cheapest thresholds are from about 0.37 to 0.50", not "exactly 0.42".

## 4. Worked example: Priya's recommendation

Priya calculates every step near the bottom of the curve, so that Omar can check it. Run the cell. You should see the totals 666, 606, 656 and 674, and about 39, 34, 28 and 19 reviews a day.

In [ ]:
rows = []
for t in [0.40, 0.42, 0.45, 0.50]:
    tp, fn, fp, tn = count(t)
    flagged = tp + fp
    rows.append([f"{t:.2f}", flagged, fn, 2 * flagged, 30 * fn, total_cost(t), round(flagged * 1.2 / 7)])
table(["Threshold", "Flagged", "Missed", "2 × flagged", "30 × missed", "Total cost", "Reviews per day"], rows)

From 0.42 to 0.45, the model saves 35 reviews (70 points) but misses 4 more urgent tickets (120 points), so the cost rises. Grace's two senior agents can review about 30 tickets a day, not 34. You use this limit in the module practice of the next lesson.

## 5. Why the best threshold moves: the costs

The next cell finds the cheapest threshold for five pairs of costs.

> **Predict.** If a missed urgent ticket costs 60 instead of 30, does the cheapest threshold go up or down? Then run the cell.

In [ ]:
def cheapest_threshold(review, miss):
    """The first threshold with the lowest total cost, and that cost."""
    costs = [total_cost(t, review, miss) for t in thresholds]
    return thresholds[costs.index(min(costs))], min(costs)

rows = []
for review, miss in [(2, 30), (2, 60), (2, 15), (4, 30), (1, 30)]:
    t, c = cheapest_threshold(review, miss)
    rows.append([review, miss, round(miss / review, 1), f"{t:.2f}", c])
table(["Review", "Miss", "Miss ÷ review", "Cheapest threshold", "Total cost"], rows, digits=1)

> **Check.** You should see 0.42 (606), 0.37 (802), 0.50 (449), 0.50 (898) and 0.37 (401). The pairs (2, 60) and (1, 30) have the same ratio, 30, and the same cheapest threshold. Only the **ratio** of the two costs decides where the bottom is.

A review costs 2 points and saves 30 points only if the ticket is urgent. So a review pays for itself when the chance that the ticket is urgent is more than 2 ÷ 30. Run the cell. You should see `6.7%`, and `3.3%` for a miss at 60.

In [ ]:
print(f"limit with a miss at 30: {2 / 30:.1%}")
print(f"limit with a miss at 60: {2 / 60:.1%}")

## 6. Why the best threshold moves: the base rate

The test set has 8% urgent tickets. The lesson's chart scales the counts to 4% and 16% urgent tickets: urgent counts are multiplied by 0.5 or 2, and the other counts are scaled so that the total stays 1,000. The next cell does the same, and rounds each cost to a whole number, as the chart does.

> **Predict.** With 16% urgent tickets, does the cheapest threshold go up or down? Then run the cell.

In [ ]:
def cost_at_base_rate(threshold, urgent, review=2, miss=30):
    """Total cost per 1,000 tickets when `urgent` of them are urgent, with the same scores."""
    tp, fn, fp, tn = count(threshold)
    wu = urgent / 80                 # scale for the urgent counts
    wn = (1000 - urgent) / 920       # scale for the other counts
    return math.floor(review * (tp * wu + fp * wn) + miss * fn * wu + 0.5)

rows = []
for urgent in [40, 80, 160]:
    costs = [cost_at_base_rate(t, urgent) for t in thresholds]
    best = thresholds[costs.index(min(costs))]
    rows.append([f"{urgent / 10:g}%", best, min(costs), costs[0], costs[-1]])
table(["Urgent", "Cheapest threshold", "Lowest cost", "Cost at 0.00", "Cost at 1.00"], rows)

> **Check.** You should see 0.50 (388) for 4%, 0.42 (606) for 8%, and 0.37 (918) for 16%. All three start at 2000 (every ticket is reviewed) and end at 1200, 2400 and 4800 (every urgent ticket is missed). With more urgent tickets, misses are a bigger part of the cost, so the cheapest threshold goes down.

## 7. Guided practice: a more expensive miss

The payment provider now charges a fee for every reversed payment. Omar says a missed urgent ticket costs **60 cost points**. A review still costs 2.

> **Your turn.** Use `thresholds`, `total_cost` and `count`. Replace each `...`, then run the check cell. Write one sentence for Grace about her team's workload in `c03-notes.md`.
>
> - `best_60`: the cheapest threshold (the first one if two cost the same).
> - `flagged_60` and `missed_60`: the flagged and the missed urgent tickets at that threshold.
> - `cost_60`: the total cost at that threshold.
> - `per_day_60`: the reviews per day (flagged × 1.2 ÷ 7).

In [ ]:
costs_60 = [total_cost(t, 2, 60) for t in thresholds]
best_60 = thresholds[costs_60.index(min(costs_60))]
tp, fn, fp, tn = count(best_60)
flagged_60 = tp + fp
missed_60 = fn
cost_60 = total_cost(best_60, 2, 60)
per_day_60 = flagged_60 * 1.2 / 7

Run the check cell.

In [ ]:
expect("the cheapest threshold", best_60, 0.37)
expect("the number of flagged tickets", flagged_60, 281)
expect("the number of missed urgent tickets", missed_60, 4)
expect("the total cost", cost_60, 802)
expect("the reviews per day", per_day_60, 48.2, tol=0.5)

The more expensive miss moved the cheapest threshold down, from 0.42 to 0.37. The senior agents would review about 48 tickets a day, about 14 more than at 0.42.

## 8. Independent variation: a more expensive review

Set the miss cost back to 30. Now a review costs **4 cost points**.

> **Predict.** Does the cheapest threshold go up or down?

> **Your turn.** Find the cheapest threshold and its total cost. Then run the check cell.

In [ ]:
costs_4 = [total_cost(t, 4, 30) for t in thresholds]
best_review_4 = thresholds[costs_4.index(min(costs_4))]
cost_review_4 = min(costs_4)

Run the check cell.

In [ ]:
expect("the cheapest threshold", best_review_4, 0.50)
expect("the total cost", cost_review_4, 898)

The ratio is now 30 ÷ 4 = 7.5, smaller than 15. A review is relatively more expensive, so the cheapest threshold goes up.

## 9. Failure case: keeping the default threshold of 0.5

Many tools turn a score into a yes or no at 0.50, because it is in the middle of 0 to 1. The tool does not know Larkfield's costs.

> **Predict.** How much more does 0.50 cost than the cheapest threshold, in percent? Then run the cell.

In [ ]:
default, best = total_cost(0.50), total_cost(0.42)
print("cost at 0.50:", default, "| cost at 0.42:", best)
print(f"0.50 costs {default / best - 1:.0%} more")
print("extra missed urgent tickets at 0.50:", count(0.50)[1] - count(0.42)[1])

> **Check.** You should see 674 and 606, that 0.50 costs 11% more, and that it misses 8 more urgent tickets. **Fix:** treat the threshold as a decision. Write down the costs, the base rate and the workload limit that you used, and plan a date to check it again.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you turned counts into a review workload (× 1.2 for a week, ÷ 7 for a day) and a total cost: 2 × flagged + 30 × missed. The cost curve over 101 thresholds has its lowest point at 0.42 (606 points), and every threshold from 0.37 to 0.50 costs less than 700. The cheapest threshold goes down when a miss costs more compared with a review, or when urgent tickets become more common.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Calibration and slices](https://learning.nextia-ai.com/courses/math/m06/calibration-and-slices/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m06/thresholds-and-costs/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.